# Week 3: Anatomy of a BLAST report

| | |
|---|---|
| **Due** | start of the Week 4 meeting |
| **Prerequisites** | Weeks 1–2 |
| **Produces** | your own BLAST result file; a score-ranking comparison |

## Learning objectives

1. Interpret every field of a BLAST hit table and say what each number does and does not establish.
2. Explain why an E-value depends on database size and a bit score does not.
3. Describe what BLAST's heuristic trades away relative to exhaustive alignment.
4. Recognise the twilight zone and say why sequence identity alone stops being decisive there.

## Where this sits

Last week you filled in a dynamic programming matrix by hand. That algorithm is exact: it is guaranteed to find the optimal alignment of two sequences. It is also quadratic — every cell of an n × m grid — which is fine for two 9-residue peptides and hopeless against a database of hundreds of millions of sequences.

BLAST is what you do instead. It is a **heuristic**: it gives up the guarantee in exchange for speed. This week is about understanding precisely what was given up, because that determines when you can trust the answer.

In [ ]:
#@title Setup — run this first
%pip install -q "biopython>=1.85" "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31"
print("setup complete")

In [ ]:
import pathlib, requests
FIXTURES = pathlib.Path("fixtures")
COURSE_DATA_URL = None   

def course_file(name: str) -> pathlib.Path:
    local = FIXTURES / name
    if local.exists():
        return local
    if COURSE_DATA_URL:
        FIXTURES.mkdir(exist_ok=True)
        r = requests.get(f"{COURSE_DATA_URL}/{name}", timeout=30); r.raise_for_status()
        local.write_bytes(r.content); return local
    raise FileNotFoundError(f"{name} not found. Put fixtures/ beside this notebook.")

print("fixtures:", FIXTURES.resolve())

---
## Part 1: Run a search

There is no fixture for this part. You need to run the search yourself, because the point of the exercise is reading output you generated and cannot anticipate.

1. Go to **https://blast.ncbi.nlm.nih.gov/Blast.cgi** and choose `protein blast`.
2. Paste the *E. coli* DHFR sequence (printed by the cell below).
3. Set the database to **UniProtKB/Swiss-Prot**.
4. Run it. This may take a minute or two.
5. When it finishes, use **Download → Hit Table (CSV)** and save the file next to this notebook as `my_blast_swissprot.csv`.

Then do it a **second** time, changing only the database to **Non-redundant protein sequences (nr)**, and save that as `my_blast_nr.csv`.

Same query, same everything else. Only the database is changing between the two searches. Save both searches!

In [ ]:
from Bio import SeqIO
ecoli = SeqIO.read(course_file("dhfr_ecoli.fasta"), "fasta")
print(f">{ecoli.id}  ({len(ecoli.seq)} residues)")
print(ecoli.seq)

### Load your results

The BLAST hit table is a headerless CSV. The column order is fixed and documented. You'll get quite familiar with it during this assignment.

In [ ]:
import pandas as pd, pathlib

COLS = ["query", "subject", "pct_identity", "align_length", "mismatches",
        "gap_opens", "q_start", "q_end", "s_start", "s_end", "evalue", "bit_score"]

def load_hits(path):
    p = pathlib.Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the BLAST search described above and download "
            "the Hit Table (CSV) to this folder."
        )
    df = pd.read_csv(p, comment="#", header=None, names=COLS)
    return df

swissprot = load_hits("my_blast_swissprot.csv")
swissprot.head(10)

### Annotate every field

For the **top three hits**, write out what each column means in your own words. 

Two of these are commonly misread:

- **`pct_identity`** is computed over the *aligned region*, not the whole query. A hit can be 100% identical over 20 residues and share nothing else.
- **`align_length`** together with the query length tells you **coverage**. A high-identity, low-coverage hit is a shared motif, not a homologous protein.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a table or markdown cell explaining all 12 fields for the top 3 hits,
#           plus a computed 'coverage' column = align_length / len(query)
# Example: for a self-hit, pct_identity 100.0, coverage 1.0, evalue near 0
# ---- END ----

---
## Part 2: The experiment

You ran the identical query against two databases of very different size.

Find a protein that appears in **both** result sets. Compare its bit score and its E-value across the two searches (like we did in class!).

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a small DataFrame comparing bit_score and evalue for the same subject
#           protein in my_blast_swissprot.csv and my_blast_nr.csv
# Example: bit_score identical (or near-identical) across the two; evalue larger in nr
# ---- END ----

**Stop and think, then write your answer below.**

The bit score barely moved. The E-value did. Why?

A **bit score** is a property of the alignment itself — a normalised measure of how good this particular pairing is, scaled so that it can be compared across searches and scoring schemes. 

An **E-value** is the number of hits this good you would *expect to see by chance* in a database of this size. Make the database bigger and you get more chances to find a good alignment by luck, so the same alignment becomes less surprising. The E-value rises even though the alignment did not change by one residue.

This has a consequence people often miss: **an E-value is not a property of your hit.** It is a property of your hit *and the database you happened to search*. When you quote an E-value, you need to also say where it came from and when.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: 3-4 sentences explaining the bit score / E-value difference in your own words
# Example: name what would happen to the E-value if the database doubled overnight
# ---- END ----

---
## Part 3: What BLAST gives up

BLAST does not fill in your matrix from last week. It:

1. Chops the query into short **words** (3 residues for protein searches).
2. Finds database sequences containing a word that scores above a threshold (i.e., "T") against one of yours — a **seed**.
3. **Extends** outward from each seed, first without gaps, then with them if the ungapped score looks promising.
4. Never looks at anything that had no qualifying seed.

Step 4 is the trade. A real homolog that happens to share no high-scoring 3-residue word with your query becomes invisible to BLAST. Importantly, it is not just scored badly but rather it is *never scored at all*.

Below, you'll see an exhaustive Smith–Waterman search over a small curated set.

In [ ]:

import pandas as pd
from Bio import SeqIO
from Bio.Align import PairwiseAligner, substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")
homologs = list(SeqIO.parse(course_file("dhfr_homologs.fasta"), "fasta"))

al = PairwiseAligner()
al.substitution_matrix = blosum62
al.mode = "local"
al.open_gap_score = -11      # affine, match BLAST's defaults
al.extend_gap_score = -1

rows = []
for rec in homologs:
    score = al.align(ecoli.seq, rec.seq).score
    rows.append({"accession": rec.id.split("|")[1],
                 "entry": rec.id.split("|")[2],
                 "length": len(rec.seq),
                 "sw_score": score})

sw = pd.DataFrame(rows).sort_values("sw_score", ascending=False).reset_index(drop=True)
sw

Two things to notice in that table.

**Three sequences score identically at the top.** `P0ABQ4`, `P0ABQ5`, and `P0ABQ6` — *E. coli* K-12, *E. coli* CFT073, and *Shigella flexneri* — are byte-for-byte the same protein. They are three database entries for one sequence. This is not an error in the fixture; it is what real databases look like, and it is the reason your BLAST results are full of near-duplicates. Remember this, because next week it will distort a conservation calculation, and later in Week 7 it explains why AlphaFold counts *effective* sequences rather than rows.

**The ranking is by score, and score scales with alignment length.** A longer protein has more opportunity to accumulate score. This is the first appearance of a confound that you'll see again in Week 13. Then, you'll find that docking scores scale with pocket size.

---
## Extensions

### Computing extension: what did BLAST miss?

Compare your BLAST ranking of the Swiss-Prot hits against an exhaustive Smith–Waterman ranking of the same sequences. Identify any sequence that Smith–Waterman scores respectably but BLAST did not report at all, and report the wall-clock difference between the two approaches.

### Biology extension: the twilight zone

Find a hit in your Swiss-Prot results at roughly **20–30% identity**. At that range, sequence identity alone stops being decisive and real homologs and unrelated proteins overlap.

Using evidence *independent of the alignment score* (e.g., annotated function, domain architecture, conserved catalytic residues, taxonomic plausibility) argue whether your hit is a genuine homolog. Then state what evidence you would need to be certain.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: your chosen extension(s), clearly labelled
# Example: for the computing extension, report timings for both approaches
# ---- END ----

---
## Submission checklist

- [ ] Both BLAST result files downloaded and loaded
- [ ] All 12 fields annotated in your own words, with a coverage column computed
- [ ] Bit score / E-value comparison across the two databases, with your written explanation
- [ ] Smith–Waterman ranking run, with a comment on the three identical top hits
- [ ] At least one extension, clearly labelled
- [ ] Notebook runs top to bottom in a fresh runtime
